# Battery RUL Model Training
**Team Kansas — IFRPM | Prem Ganesh**

Trains XGBoost and LSTM models on the NASA Battery Degradation dataset to predict Remaining Useful Life (RUL).
Evaluation uses RMSE, MAE, and the NASA asymmetric scoring function (consistent with team convention).

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import json
import warnings
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import xgboost as xgb
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pickle
import os

warnings.filterwarnings('ignore')
np.random.seed(42)
torch.manual_seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# Output directory
os.makedirs('outputs', exist_ok=True)

## 2. Load & Prepare Data

In [ ]:
df = pd.read_csv('battery_rul_features.csv')
print(f'Total rows: {len(df)} | Batteries: {df["battery_id"].nunique()}')
print(f'Columns: {df.columns.tolist()}')

In [ ]:
# Remove censored cycles and known anomalous batteries
ANOMALOUS = ['B0033', 'B0041', 'B0050']
df_clean = df[
    (~df['is_censored']) &
    (~df['battery_id'].isin(ANOMALOUS)) &
    (df['SOH'] > 0) & (df['SOH'] <= 110)
].copy()

print(f'After cleaning: {len(df_clean)} rows | {df_clean["battery_id"].nunique()} batteries')

FEATURES = [
    'ambient_temperature', 'capacity_ahr', 'SOH',
    'capacity_fade_rate', 'SOH_rolling_mean', 'SOH_rolling_std',
    'temperature_stress_factor', 'cycle_normalized', 'Re_norm', 'Rct_norm'
]
TARGET = 'RUL'

df_model = df_clean[FEATURES + [TARGET, 'battery_id']].dropna()
print(f'After dropping NaN: {len(df_model)} rows | {df_model["battery_id"].nunique()} batteries')

In [ ]:
# Battery-level train/test split (80/20) — no data leakage
batteries = df_model['battery_id'].unique()
np.random.shuffle(batteries)
n_test = max(1, int(len(batteries) * 0.2))
test_bats  = batteries[:n_test]
train_bats = batteries[n_test:]

print(f'Train: {len(train_bats)} batteries | Test: {len(test_bats)} batteries')
print(f'Test batteries: {sorted(test_bats)}')

train_df = df_model[df_model['battery_id'].isin(train_bats)].copy()
test_df  = df_model[df_model['battery_id'].isin(test_bats)].copy()
print(f'Train samples: {len(train_df)} | Test samples: {len(test_df)}')

## 3. Evaluation Metrics

In [ ]:
def nasa_score(y_true, y_pred):
    """NASA asymmetric scoring function — penalises late predictions more than early.
    Consistent with team convention in rul_prediction.py and lstm_rul.py."""
    d = np.array(y_pred) - np.array(y_true)
    score = np.where(d < 0, np.exp(-d / 13.0) - 1, np.exp(d / 10.0) - 1)
    return float(np.sum(score))

def evaluate(y_true, y_pred, label=''):
    rmse  = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae   = float(mean_absolute_error(y_true, y_pred))
    nasa  = nasa_score(y_true, y_pred)
    r2    = float(1 - np.sum((np.array(y_true) - np.array(y_pred))**2) /
                      np.sum((np.array(y_true) - np.mean(y_true))**2))
    if label:
        print(f'[{label}] RMSE: {rmse:.3f} | MAE: {mae:.3f} | NASA Score: {nasa:.1f} | R²: {r2:.4f}')
    return {'rmse': rmse, 'mae': mae, 'nasa_score': nasa, 'r2': r2}

## 4. XGBoost Model

In [ ]:
scaler_xgb = MinMaxScaler()
X_train = scaler_xgb.fit_transform(train_df[FEATURES])
X_test  = scaler_xgb.transform(test_df[FEATURES])
y_train = train_df[TARGET].values
y_test  = test_df[TARGET].values

xgb_model = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    gamma=0.1,
    random_state=42,
    n_jobs=-1,
    verbosity=0
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50
)

xgb_preds = xgb_model.predict(X_test)
xgb_metrics = evaluate(y_test, xgb_preds, 'XGBoost')

In [ ]:
# Feature importance
importances = xgb_model.feature_importances_
sorted_idx  = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(range(len(FEATURES)), importances[sorted_idx], color='steelblue', edgecolor='white')
ax.set_xticks(range(len(FEATURES)))
ax.set_xticklabels([FEATURES[i] for i in sorted_idx], rotation=40, ha='right', fontsize=10)
ax.set_ylabel('F-score Importance')
ax.set_title('XGBoost Feature Importance — Battery RUL')
for bar, val in zip(bars, importances[sorted_idx]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{val:.3f}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig('outputs/xgb_feature_importance.png', dpi=150)
plt.show()
print('Saved: outputs/xgb_feature_importance.png')

In [ ]:
# Predicted vs Actual RUL trajectory per test battery
test_df = test_df.copy()
test_df['xgb_pred'] = xgb_preds

n_bats = len(test_bats)
cols = min(3, n_bats)
rows = (n_bats + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
axes = np.array(axes).flatten()

for i, bat in enumerate(sorted(test_bats)):
    sub = test_df[test_df['battery_id'] == bat].sort_values('cycle_normalized')
    axes[i].plot(sub['cycle_normalized'], sub['RUL'], label='Actual', color='steelblue', linewidth=2)
    axes[i].plot(sub['cycle_normalized'], sub['xgb_pred'], label='Predicted', color='coral',
                 linewidth=2, linestyle='--')
    axes[i].set_title(f'{bat}', fontsize=11)
    axes[i].set_xlabel('Cycle (normalized)')
    axes[i].set_ylabel('RUL (cycles)')
    axes[i].legend(fontsize=8)
    axes[i].grid(True, alpha=0.3)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('XGBoost — Predicted vs Actual RUL (Test Batteries)', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('outputs/xgb_rul_trajectories.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/xgb_rul_trajectories.png')

## 5. LSTM Model

In [ ]:
WINDOW = 30
RUL_CAP = int(df_model['RUL'].max())
print(f'Sequence window: {WINDOW} cycles | RUL cap: {RUL_CAP}')

scaler_lstm = MinMaxScaler()
# Fit on train features only
train_df_lstm = train_df.copy()
test_df_lstm  = test_df.copy()
train_df_lstm[FEATURES] = scaler_lstm.fit_transform(train_df_lstm[FEATURES])
test_df_lstm[FEATURES]  = scaler_lstm.transform(test_df_lstm[FEATURES])

def build_sequences(data, features, target, window, rul_cap):
    """Sliding window per battery, piecewise-linear RUL cap (same as team lstm_rul.py)."""
    X_list, y_list = [], []
    for bat, grp in data.groupby('battery_id'):
        grp = grp.sort_values('cycle_normalized').reset_index(drop=True)
        feat_arr = grp[features].values
        rul_arr  = np.minimum(grp[target].values, rul_cap).astype(np.float32)
        for i in range(len(grp) - window + 1):
            X_list.append(feat_arr[i:i+window])
            y_list.append(rul_arr[i + window - 1])
    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.float32)

X_tr, y_tr = build_sequences(train_df_lstm, FEATURES, TARGET, WINDOW, RUL_CAP)
X_te, y_te = build_sequences(test_df_lstm,  FEATURES, TARGET, WINDOW, RUL_CAP)
print(f'Train sequences: {X_tr.shape} | Test sequences: {X_te.shape}')

In [ ]:
class RULDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y).unsqueeze(1)
    def __len__(self):  return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

train_loader = DataLoader(RULDataset(X_tr, y_tr), batch_size=32, shuffle=True)
test_loader  = DataLoader(RULDataset(X_te, y_te), batch_size=32, shuffle=False)


class LSTMModel(nn.Module):
    """2-layer stacked LSTM → FC, consistent with team lstm_rul.py architecture."""
    def __init__(self, input_size, hidden1=128, hidden2=64, dropout=0.3):
        super().__init__()
        self.lstm1 = nn.LSTM(input_size, hidden1, batch_first=True)
        self.drop1  = nn.Dropout(dropout)
        self.lstm2 = nn.LSTM(hidden1, hidden2, batch_first=True)
        self.drop2  = nn.Dropout(dropout)
        self.fc1   = nn.Linear(hidden2, 32)
        self.relu  = nn.ReLU()
        self.fc2   = nn.Linear(32, 1)

    def forward(self, x):
        out, _ = self.lstm1(x)
        out = self.drop1(out)
        out, _ = self.lstm2(out)
        out = self.drop2(out[:, -1, :])  # last timestep
        out = self.relu(self.fc1(out))
        return self.fc2(out)


model = LSTMModel(input_size=len(FEATURES)).to(DEVICE)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params:,}')

In [ ]:
EPOCHS    = 100
PATIENCE  = 10
LR        = 1e-3

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5, verbose=True)

best_val_loss  = float('inf')
patience_count = 0
train_losses, val_losses = [], []

for epoch in range(1, EPOCHS + 1):
    # Train
    model.train()
    batch_losses = []
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
        optimizer.zero_grad()
        pred = model(X_b)
        loss = criterion(pred, y_b)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # gradient clipping
        optimizer.step()
        batch_losses.append(loss.item())
    train_loss = np.mean(batch_losses)

    # Validate
    model.eval()
    val_batch_losses = []
    with torch.no_grad():
        for X_b, y_b in test_loader:
            X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
            pred = model(X_b)
            val_batch_losses.append(criterion(pred, y_b).item())
    val_loss = np.mean(val_batch_losses)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_count = 0
        torch.save(model.state_dict(), 'outputs/battery_lstm_best.pt')
    else:
        patience_count += 1

    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Patience: {patience_count}/{PATIENCE}')

    if patience_count >= PATIENCE:
        print(f'Early stopping at epoch {epoch}')
        break

print(f'Best val loss: {best_val_loss:.4f}')

In [ ]:
# Loss curve
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(train_losses, label='Train Loss', color='steelblue', linewidth=2)
ax.plot(val_losses,   label='Val Loss',   color='coral',     linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('LSTM Training Loss Curve — Battery RUL')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/lstm_loss_curve.png', dpi=150)
plt.show()
print('Saved: outputs/lstm_loss_curve.png')

In [ ]:
# Load best weights and evaluate
model.load_state_dict(torch.load('outputs/battery_lstm_best.pt', map_location=DEVICE))
model.eval()

lstm_preds_list = []
with torch.no_grad():
    for X_b, _ in test_loader:
        lstm_preds_list.append(model(X_b.to(DEVICE)).cpu().numpy().flatten())

lstm_preds = np.concatenate(lstm_preds_list)
lstm_metrics = evaluate(y_te, lstm_preds, 'LSTM')

In [ ]:
# LSTM predicted vs actual scatter
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_te, lstm_preds, alpha=0.4, color='steelblue', s=15, label='Predictions')
lims = [min(y_te.min(), lstm_preds.min()), max(y_te.max(), lstm_preds.max())]
ax.plot(lims, lims, 'r--', linewidth=2, label='Perfect prediction')
ax.set_xlabel('Actual RUL (cycles)')
ax.set_ylabel('Predicted RUL (cycles)')
ax.set_title(f'LSTM — Actual vs Predicted RUL\nRMSE={lstm_metrics["rmse"]:.2f} | R²={lstm_metrics["r2"]:.3f}')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/lstm_scatter.png', dpi=150)
plt.show()

## 6. Model Comparison

In [ ]:
results = {
    'dataset': 'NASA Battery Degradation',
    'train_batteries': sorted(train_bats.tolist()),
    'test_batteries':  sorted(test_bats.tolist()),
    'features': FEATURES,
    'rul_cap': RUL_CAP,
    'window_size': WINDOW,
    'eol_threshold_ahr': 1.4,
    'xgboost': xgb_metrics,
    'lstm': lstm_metrics
}

with open('outputs/battery_model_metrics.json', 'w') as f:
    json.dump(results, f, indent=2)

print('Metrics saved to outputs/battery_model_metrics.json')

# Summary table
summary = pd.DataFrame({
    'Model':      ['XGBoost', 'LSTM'],
    'RMSE':       [xgb_metrics['rmse'],       lstm_metrics['rmse']],
    'MAE':        [xgb_metrics['mae'],        lstm_metrics['mae']],
    'NASA Score': [xgb_metrics['nasa_score'], lstm_metrics['nasa_score']],
    'R²':         [xgb_metrics['r2'],         lstm_metrics['r2']]
})
summary = summary.set_index('Model').round(4)
print('\n=== Model Comparison ===')
print(summary.to_string())

In [ ]:
# Comparison bar chart
metrics_plot = ['RMSE', 'MAE', 'R²']
xgb_vals  = [xgb_metrics['rmse'],  xgb_metrics['mae'],  xgb_metrics['r2']]
lstm_vals = [lstm_metrics['rmse'], lstm_metrics['mae'], lstm_metrics['r2']]

x = np.arange(len(metrics_plot))
width = 0.35
fig, ax = plt.subplots(figsize=(8, 5))
b1 = ax.bar(x - width/2, xgb_vals,  width, label='XGBoost', color='steelblue',  edgecolor='white')
b2 = ax.bar(x + width/2, lstm_vals, width, label='LSTM',    color='darkorange', edgecolor='white')

for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(metrics_plot, fontsize=12)
ax.set_title('XGBoost vs LSTM — Battery RUL Prediction', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/model_comparison.png', dpi=150)
plt.show()
print('Saved: outputs/model_comparison.png')

## 7. Save Models

In [ ]:
# XGBoost
with open('outputs/battery_xgb_model.pkl', 'wb') as f:
    pickle.dump({'model': xgb_model, 'scaler': scaler_xgb, 'features': FEATURES}, f)
print('Saved: outputs/battery_xgb_model.pkl')

# LSTM — best weights already saved during training
# Also save scaler and config for inference
with open('outputs/battery_lstm_config.pkl', 'wb') as f:
    pickle.dump({
        'scaler': scaler_lstm,
        'features': FEATURES,
        'window': WINDOW,
        'rul_cap': RUL_CAP,
        'input_size': len(FEATURES)
    }, f)
print('Saved: outputs/battery_lstm_config.pkl')
print('\nAll outputs saved to outputs/')